# convexity.ipynb

- Reference: https://aubreymoore.github.io/jb/create-one-to-many-sql/

## TODO

- [x] move `get_data_for_images_table` function to `roadside.py`
- [x] move `get_data_for_detections_table` function to `roadside.py`
- [x] remove detection attribute fields from schema
- [x] prevent duplication of records in database
- [x] add fields to `detections` table
- [] handle images which do not contain detected objects
- [x] change filename of this notebook
- [] segmentation results saved in `~Desktop/blog2026` - why??
- [] find second weights file

In [1]:
%matplotlib widget

import roadside as rs
import sqlite3
import os
from icecream import ic
import pandas as pd
import math
import numpy as np
from shapely.wkt import loads as wkt_loads
import cv2

# reference: https://kioku-space.com/en/jupyter-skip-execution/
from IPython.core.magic import register_cell_magic

@register_cell_magic
def skip(line, cell):
    return

# Functions

In [2]:
import sqlite3
import os # Optional: use os.remove() to clean up test file

def create_db(db_path: str) -> None:
    """ Creates an SQLite3 database"""

    # schema SQL script as a string
    sql_script = """
--- 2026-03-31 17:30

CREATE TABLE IF NOT EXISTS images (
  image_id INTEGER PRIMARY KEY,
  image_path TEXT UNIQUE,
  image_width INTEGER,
  image_height INTEGER,
  timestamp TEXT,
  latitude REAL,
  longitude REAL
);

CREATE TABLE IF NOT EXISTS detections (
  detection_id INTEGER PRIMARY KEY,
  image_id INTEGER,
  class_id INTEGER,
  poly_wkt TEXT,
  poly_wkt_c TEXT,
  x_min INTEGER,
  y_min INTEGER,
  x_max INTEGER,
  y_max INTEGER,
  confidence REAL,
  is_accepted INTEGER NOT NULL DEFAULT 0,
  is_healthy INTEGER NOT NULL DEFAULT 0,
  is_damaged_with_vcuts INTEGER NOT NULL DEFAULT 0,
  is_damaged_without_vcuts INTEGER NOT NULL DEFAULT 0,
  is_dead INTEGER NOT NULL DEFAULT 0,
  is_rejected INTEGER NOT NULL DEFAULT 0,
  is_crowded INTEGER NOT NULL DEFAULT 0,
  is_occluded INTEGER NOT NULL DEFAULT 0,
  has_other_problem INTEGER NOT NULL DEFAULT 0,
  FOREIGN KEY(image_id) REFERENCES images(image_id) ON DELETE CASCADE 
);

CREATE TABLE IF NOT EXISTS vcuts (
  vcut_id INTEGER PRIMARY KEY,
  detection_id INTEGER,
  start_x INTEGER,
  start_y INTEGER,
  far_x INTEGER,
  far_y INTEGER,        
  end_x INTEGER,
  end_y INTEGER,
  depth REAL,
  degrees REAL,
  FOREIGN KEY(detection_id) REFERENCES detections(detection_id) ON DELETE CASCADE
);
    """
    
    # Optional: Remove the database file if it already exists for a clean run
    if os.path.exists(db_path):
        os.remove(db_path)

    try:
        # Establish a connection (creates the DB file if it doesn't exist)
        conn = sqlite3.connect(db_path)
        print(f"Database {db_path} created/opened.")

        # Create a cursor object
        cursor = conn.cursor()

        # Execute the entire SQL script from the string
        cursor.executescript(sql_script)
        print("SQL script executed successfully.")

        # Commit the changes
        conn.commit()
        print("Changes committed.")

    except sqlite3.Error as e:
        print(f"An error occurred: {e}")
        if conn:
            conn.rollback() # Roll back changes if an error occurs

    finally:
        # Close the connection
        if conn:
            conn.close()
            print("\nDatabase connection closed.")

# Example usage:
# create_db("example.db")

In [3]:
import gc
import torch

def delete_results_from_gpu_memory():
    """
    Explicitly manages memory after processing each image to prevent running out of GPU memory
    """
    global results_gpu
    del results_gpu 
    gc.collect() 
    torch.cuda.empty_cache() # Clears unoccupied cached memory

# Usage example:
    
# delete_results_from_gpu_memory()


In [4]:
def get_data_for_vcuts_table(db_path, image_id:int)->pd.DataFrame:
    """ 
    Processes data for a single image. 
    Returns a pandas dataframe containing data to be added to the `vcuts` table.
    """
    # get input data from the images and detections tables
    sql = """ 
    SELECT image_path, detection_id, poly_wkt
    FROM images, detections
    WHERE images.image_id = detections.image_id
    """
    df_input = pd.read_sql(sql, sqlite3.connect(db_path))
    ic(df_input)

    data_list = [] # list of dicts to be converted to dataframe for insertion into vcuts table
    for i, r in df_input.iterrows():

        tree_contour = np.array(
            list(wkt_loads(r.poly_wkt).exterior.coords), 
            dtype=np.int32).reshape(-1, 1, 2)
        hull_indices = cv2.convexHull(tree_contour, returnPoints=False)
        # ic(i, hull_indices)
        defects = cv2.convexityDefects(tree_contour, hull_indices)
        # ic(defects)
        # ic(defects.shape)
        
        # process defects
        for j, defect in enumerate(defects):
            
            start_idx, end_idx, far_idx, depth = defect[0]
            start_point = tree_contour[start_idx][0]
            end_point = tree_contour[end_idx][0]
            far_point = tree_contour[far_idx][0]
            
            # calculate length of all sides of triangle in pixels
            a = math.sqrt((end_point[0] - start_point[0])**2 + (end_point[1] - start_point[1])**2)
            b = math.sqrt((far_point[0] - start_point[0])**2 + (far_point[1] - start_point[1])**2)
            c = math.sqrt((end_point[0] - far_point[0])**2 + (end_point[1] - far_point[1])**2)
            
            # calculate depth of the convexity defect using Heron's formula in pixels
            # equals distance between far_point point and base of the triangle (the `a` side) 
            s = (a+b+c)/2
            area = math.sqrt(s*(s-a)*(s-b)*(s-c))
            depth = (2*area)/a # depth is the height of the triangle, which is the distance from the far point to the line formed by the start and end points
            
            # calculate the angle of the defect in degrees using the cosine rule
            angle = math.degrees(math.acos((b**2 + c**2 - a**2)/(2*b*c)))
            # ic(i, d, angle, start, far, end)

            data_dict = {
                'detection_id': r.detection_id,
                'start_x': start_point[0],
                'start_y': start_point[1],
                'far_x': far_point[0],
                'far_y': far_point[1],
                'end_x': end_point[0],
                'end_y': end_point[1],
                'depth': depth,
                'degrees': angle
            }
            data_list.append(data_dict)
        df_output = pd.DataFrame(data_list)
    return df_output

# # Usage example:
# df_vcuts = get_data_for_vcuts_table(db_path=db_path, image_id=0)
# df_vcuts.to_sql('vcuts', sqlite3.connect(db_path), if_exists='delete_rows', index=False)

# Global variables for this run

In [5]:
root_dir = "/home/aubrey/Desktop/sam3-2026-01-31"
image_paths = ["example_images/08hs-palms-03-zglw-superJumbo.webp"]
text_prompts=["coconut palm tree"]
db_path = 'sam3_detections.sqlite3'

# Main

In [6]:


assert rs.check_gpu(), 'ERROR: GPU is unavailable.'

create_db(db_path)
con = sqlite3.connect(db_path)

# scan images and populate database
for image_path in image_paths:
    
    # skip image if it is already in the database
    if con.execute(f'SELECT COUNT(*) FROM images WHERE  image_path = "{image_path}"').fetchone()[0] > 0:
        print(f'WARNING: Image {image_path} is already in the database. Skipping to next image.')
        continue
        
    # Detect objects in image
    results_gpu = rs.run_sam3_semantic_predictor(input_image_path=image_path, text_prompts=text_prompts)
    
    # Free up GPU memory in preparation for detecting objects in the next image
    # This is a work-around to prevent out-of-memory errors from the GPU
    # I move all results for further processing and use the GPU only for object detection.
    results_cpu = [r.cpu() for r in results_gpu] # copy results to CPU
    print('deleting results from GPU memory')       
    delete_results_from_gpu_memory() # Clear GPU memory after processing each image
    
    # populate 'images' table
    image_width, image_height, timestamp, latitude, longitude = rs.get_data_for_images_table(results_cpu, image_path)
    sql = """
    INSERT INTO images (image_path, image_width, image_height, timestamp, latitude, longitude) 
    VALUES (?,?,?,?,?,?)
    RETURNING image_id
    """
    parameters = (image_path, image_width, image_height, timestamp, latitude, longitude,)
    
    # sql = 'INSERT INTO images (image_path) VALUES (?) RETURNING image_id'
    try:
        image_id = con.execute(sql, parameters).fetchone()[0] # THE COMMA IN THE PARAMETERS TUPLE IS IMPORTANT
    except sqlite3.IntegrityError as e:
        print(f'ERROR: Image {image_path} already exists in {db_path}')
        raise e    
    con.commit()
    ic(image_id)
    
    # populate 'detections' table
    # df_detections = rs.get_data_for_detections_table(results_cpu, image_id)
    df_detections = rs.get_data_for_detections_table(results_cpu, image_id)
    for i, r in df_detections.iterrows():
            # populate 'detections' table
        sql = ''' 
        INSERT INTO detections
            (image_id, class_id, poly_wkt, poly_wkt_c, x_min, y_min, x_max, y_max, confidence)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?);
        '''
        parameters = (image_id, 0, r['poly_wkt'], r['poly_wkt_c'], r['x_min'], r['y_min'], r['x_max'], r['y_max'], r['confidence']) 
        con.execute(sql, parameters)
        con.commit()
        
    # scan the 'detections' table for vcuts and populate 'vcuts' table  
    df_vcuts = get_data_for_vcuts_table(db_path=db_path, image_id=0)
    df_vcuts.to_sql('vcuts', sqlite3.connect(db_path), if_exists='delete_rows', index=False)
    
con.close()   
print('FINISHED')    

CUDA version: 13.0
GPU device name: NVIDIA GeForce RTX 3080 Laptop GPU
Database sam3_detections.sqlite3 created/opened.
SQL script executed successfully.
Changes committed.

Database connection closed.
Ultralytics 8.4.26 🚀 Python-3.13.11 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 3080 Laptop GPU, 15982MiB)

image 1/1 /home/aubrey/Desktop/temp/example_images/08hs-palms-03-zglw-superJumbo.webp: 1932x1932 2 coconut palm trees, 1343.5ms
Speed: 13.2ms preprocess, 1343.5ms inference, 16.9ms postprocess per image at shape (1, 3, 1932, 1932)
Results saved to /home/aubrey/Desktop/blog2026/runs/segment/predict48
deleting results from GPU memory
FINISHED


ic| image_id: 1
ic| df_input:                                           image_path  detection_id  \
              0  example_images/08hs-palms-03-zglw-superJumbo.webp             1   
              1  example_images/08hs-palms-03-zglw-superJumbo.webp             2   
              
                                                          poly_wkt  
              0  POLYGON ((486 679, 486 684, 485 685, 485 688, ...  
              1  POLYGON ((1035 1274, 1035 1276, 1034 1277, 103...  


# Retrieve data from database

In [7]:
pd.read_sql('SELECT * FROM images', sqlite3.connect(db_path))

,image_id,image_path,image_width,image_height,timestamp,latitude,longitude
0,1,example_images/08hs-palms-03-zglw-superJumbo.webp,1366,2048,None,None,None


In [8]:
pd.read_sql('SELECT * FROM detections', sqlite3.connect(db_path))

,detection_id,image_id,class_id,poly_wkt,poly_wkt_c,x_min,y_min,x_max,y_max,confidence,is_accepted,is_healthy,is_damaged_with_vcuts,is_damaged_without_vcuts,is_dead,is_rejected,is_crowded,is_occluded,has_other_problem
0,1,1,0,"POLYGON ((486 679, 486 684, 485 685, 485 688, ...","POLYGON ((486 1369, 486 1364, 485 1363, 485 13...",354,683,593,2068,0.928711,0,0,0,0,0,0,0,0,0
1,2,1,0,"POLYGON ((1035 1274, 1035 1276, 1034 1277, 103...","POLYGON ((1035 774, 1035 772, 1034 771, 1034 7...",701,1273,1103,2066,0.920898,0,0,0,0,0,0,0,0,0


In [9]:
pd.read_sql('SELECT * FROM vcuts', sqlite3.connect(db_path))

,vcut_id,detection_id,start_x,start_y,far_x,far_y,end_x,end_y,depth,degrees
0,1,1,486,679,485,688,479,695,2.691212,145.738897
1,2,1,479,695,479,699,478,700,0.784465,135.000000
2,3,1,478,700,505,841,451,849,51.708237,87.586664
3,4,1,451,849,451,854,450,855,0.821995,135.000000
4,5,1,450,855,524,966,355,2032,82.690304,137.301429
5,6,1,355,2044,357,2046,357,2047,0.554700,135.000000
6,7,1,357,2047,387,2045,402,2047,2.000000,168.591282
7,8,1,402,2047,402,2046,405,2043,0.600000,135.000000
8,9,1,405,2043,405,2040,406,2039,0.727607,135.000000
9,10,1,406,2039,406,2034,408,2028,0.894427,161.565051


In [10]:
import cv2
import matplotlib.pyplot as plt
from shapely.wkt import loads
import numpy as np

image_path = df_images['image_path'][0]
img =cv2.imread(image_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

poly_wkt = df_detections['poly_wkt'][0]
# use shapely to parse WKT and extract coordinates
polygon = loads(poly_wkt)
# convert to OpenCV format (numpy array of shape (n, 1, 2) with dtype=np.int32)
coords = np.array(list(polygon.exterior.coords), dtype=np.int32)
pts = coords.reshape((-1, 1, 2))
# Draw the polygon
cv2.polylines(img, [pts], isClosed=True, color=(0, 255, 0), thickness=2)

plt.imshow(img)
plt.show() # important: forces the display and resets the image state

NameError: name 'df_images' is not defined

In [ ]:
hull_indices = cv2.convexHull(pts, returnPoints=False)
# ic(hull_indices)
defects = cv2.convexityDefects(pts, hull_indices)
ic(defects);
ic(defects.shape)

# 1. Extract the distance column (index 3)
# defects shape is (N, 1, 4), so [:, 0, 3] gets the distances
distances = defects[:, 0, 3]

# argsort returns ascending indices; [::-1] reverses it for descending
sort_indices = np.argsort(distances)[::-1]

# 3. Reorder the original defects array
defects = defects[sort_indices]
ic(defects)

# segs = []
# for i in range(10):
#     s,e,f,d = defects[i,0]
#     start = tuple(pts[s][0])
#     end = tuple(pts[e][0])
#     far = tuple(pts[f][0])
#     cnt_segs.append({'start': start, 'end': end, 'far': far})


for i in range(defects.shape[0]):
    s,e,f,d = defects[i,0]
    start = tuple(pts[s][0])
    end = tuple(pts[e][0])
    far = tuple(pts[f][0])
    # pt= (100,180)
    
    # find length of all sides of triangle
    a = math.sqrt((end[0] - start[0])**2 + (end[1] - start[1])**2)
    b = math.sqrt((far[0] - start[0])**2 + (far[1] - start[1])**2)
    c = math.sqrt((end[0] - far[0])**2 + (end[1] - far[1])**2)
    s = (a+b+c)/2
    ar = math.sqrt(s*(s-a)*(s-b)*(s-c))
    
    #distance between point and convex hull
    d=(2*ar)/a
    
    # apply cosine rule here
    # angle = math.acos((b**2 + c**2 - a**2)/(2*b*c)) * 57
    angle = math.degrees(math.acos((b**2 + c**2 - a**2)/(2*b*c)))
    ic(i, d, angle, start, far, end)

# hull_points = cv2.convexHull(pts, returnPoints=True)
# cv2.drawContours(img, [hull_points], -1, (255, 0, 0), 2) # Draws in red with thickness 2
# plt.imshow(img)
# plt.show()

In [ ]:
image_path = df_images['image_path'][0]
img =cv2.imread(image_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

#1
points = np.array([[478,700], [505,841], [451,849]], np.int32)
points = points.reshape((-1, 1, 2))
cv2.polylines(img, [points], False, (255, 0, 0), 2) # Color is RED, thickness is 2

#3
points = np.array([[579,836], [547,820], [563, 767]], np.int32)
points = points.reshape((-1, 1, 2))
cv2.polylines(img, [points], False, (255, 0, 0), 2) # Color is RED, thickness is 2

plt.imshow(img)
plt.show()

In [ ]:
poly_wkt_to_np_array = lambda poly_wkt: np.array(list(loads(poly_wkt).exterior.coords), dtype=np.int32)

poly_wkt = df_detections['poly_wkt'][0]
# use shapely to parse WKT and extract coordinates
polygon = loads(poly_wkt)
# convert to OpenCV format (numpy array of shape (n, 1, 2) with dtype=np.int32)
coords = np.array(list(polygon.exterior.coords), dtype=np.int32)
pts = coords.reshape((-1, 1, 2))

In [ ]:
sql = """ 
SELECT image_path, poly_wkt
FROM images, detections
WHERE images.image_id = detections.image_id
"""
df = pd.read_sql(sql, sqlite3.connect(db_path))

poly_wkt_to_np_array = lambda poly_wkt: np.array(list(loads(poly_wkt).exterior.coords), dtype=np.int32)

df['poly_coords'] = df['poly_wkt'].apply(poly_wkt_to_np_array)

df


In [ ]:
img = cv2.imread(df['image_path'][0])
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  
pts = df['poly_coords'][1].reshape((-1, 1, 2))
cv2.polylines(img, [pts], isClosed=True, color=(0, 255, 0), thickness=2)
plt.imshow(img)
plt.show()